<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** For an editor with limited review capacity, can a leakage-aware model rank pages in the observed declining-trend category more effectively than a transparent stale-and-visible rule?

**Decision supported:** which existing pages should be manually investigated first.

**Who acts:** a content editor or SEO lead.

**Cost of a wrong call:** a false positive wastes editorial time; a false negative may delay investigation of a visible page. The model is therefore evaluated at the top of the queue with Precision@50, not only with general classification metrics.


In [ ]:
from pathlib import Path
import os, urllib.request
DATA_NAME = "content_refresh_anonymized.csv"
HERE = Path.cwd().resolve()
data_path = next((p / "data" / "raw" / DATA_NAME for p in [HERE, *HERE.parents] if (p / "data" / "raw" / DATA_NAME).exists()), None)
if data_path is None:
    ROOT = HERE / "flyrank_notebook_workspace"
    data_path = ROOT / "data" / "raw" / DATA_NAME
    data_path.parent.mkdir(parents=True, exist_ok=True)
    if not data_path.exists():
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv", data_path)
        except Exception as exc:
            raise FileNotFoundError("Extract the internship ZIP and open this notebook from it, or provide the public starter CSV.") from exc
else:
    ROOT = data_path.parents[2]
os.chdir(ROOT)
OUT = ROOT / "work" / "outputs"; OUT.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "work" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
print("Dataset:", data_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv(data_path)
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
for raw in ["impressions_90d", "clicks_90d", "sessions_90d"]:
    df[f"log_{raw}"] = np.log1p(pd.to_numeric(df[raw], errors="coerce").clip(lower=0))

numeric = [c for c in ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update", "avg_position", "ctr", "engagement_rate", "scroll_rate", "word_count", "char_count"] if c in df]
categorical = [c for c in ["content_type", "main_intent", "freshness_tier", "position_tier"] if c in df]
X, y, groups = df[numeric + categorical], df["is_declining_label"], df["client_id"].astype(str)
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42).split(X, y, groups=groups))

preprocess = ColumnTransformer([
    ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
    ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])
def p_at_k(y_true, scores, k):
    return float(np.asarray(y_true)[np.argsort(-np.asarray(scores))[:k]].mean())

models = {
    "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=25, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
}
fitted, rows = {}, []
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", estimator)]).fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = pipe.predict_proba(X.iloc[test_idx])[:, 1]
    fitted[name] = (pipe, scores)
    rows.append({"model":name, "roc_auc":roc_auc_score(y.iloc[test_idx], scores), "average_precision":average_precision_score(y.iloc[test_idx], scores), "precision_at_20":p_at_k(y.iloc[test_idx], scores, 20), "precision_at_50":p_at_k(y.iloc[test_idx], scores, 50)})

baseline_score = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(float) * np.log1p(df["impressions_90d"].clip(lower=0))
rows.insert(0, {"model":"transparent_stale_visible_baseline", "roc_auc":roc_auc_score(y.iloc[test_idx], baseline_score.iloc[test_idx]), "average_precision":average_precision_score(y.iloc[test_idx], baseline_score.iloc[test_idx]), "precision_at_20":p_at_k(y.iloc[test_idx], baseline_score.iloc[test_idx], 20), "precision_at_50":p_at_k(y.iloc[test_idx], baseline_score.iloc[test_idx], 50)})
results = pd.DataFrame(rows)
best_name = results.loc[results.model != "transparent_stale_visible_baseline"].sort_values("precision_at_50", ascending=False).iloc[0].model
best_model, _ = fitted[best_name]
print("Rows:", len(df), "| clients:", groups.nunique(), "| held-out clients:", groups.iloc[test_idx].nunique(), "| held-out base rate:", round(y.iloc[test_idx].mean(), 3))


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
